In [0]:
# Databricks notebook source

# COMMAND ----------
# Configuración

CATALOG_NAME   = "workspace"
SCHEMA_NAME    = "default"

SILVER         = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_retail_sales"
DIM_CUSTOMER   = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_customer"
DIM_ITEM       = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_item"
DIM_DATE       = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_date"
FACT_SALES     = f"{CATALOG_NAME}.{SCHEMA_NAME}.fact_sales"

# COMMAND ----------
# Lectura de tablas

from pyspark.sql import functions as F

silver       = spark.table(SILVER)
dim_customer = spark.table(DIM_CUSTOMER).select("customer_key", "customer_id")
dim_item     = spark.table(DIM_ITEM).select("item_key", "item_name")
dim_date     = spark.table(DIM_DATE).select("date_key", "full_date")

count_silver = silver.count()
print(f"Silver         : {count_silver:,} filas")
print(f"dim_customer   : {dim_customer.count():,} filas")
print(f"dim_item       : {dim_item.count():,} filas")
print(f"dim_date       : {dim_date.count():,} filas")

# COMMAND ----------
# Construcción de fact_sales

fact = (
    silver
    # Join con dim_customer
    .join(dim_customer, on="customer_id", how="left")
    # Join con dim_item: silver.item = dim_item.item_name
    .join(dim_item, silver["item"] == dim_item["item_name"], how="left")
    # Join con dim_date: silver.transaction_date = dim_date.full_date
    .join(dim_date, silver["transaction_date"] == dim_date["full_date"], how="left")
    .select(
        F.col("transaction_id").cast("string"),
        F.col("customer_key").cast("bigint"),
        F.col("item_key").cast("bigint"),
        F.col("date_key").cast("int"),
        F.col("quantity").cast("int"),
        F.col("price_per_unit").cast("decimal(10,2)"),
        F.col("total_spent").cast("decimal(12,2)"),
        F.col("payment_method").cast("string"),
        F.col("location").cast("string"),
        F.col("discount_applied").cast("boolean"),
    )
)

# COMMAND ----------
# Escritura fact_sales

(
    fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_SALES)
)

print(f"✓ Tabla escrita: {FACT_SALES}")

# COMMAND ----------
# Validación 1 — conteo Silver vs fact_sales

count_fact = spark.table(FACT_SALES).count()
print(f"Filas Silver    : {count_silver:,}")
print(f"Filas fact_sales: {count_fact:,}")
assert count_fact == count_silver, f"Discrepancia: {abs(count_fact - count_silver)} filas"
print("✓ Conteo OK")

# COMMAND ----------
# Validación 2 — transaction_id único

total     = spark.table(FACT_SALES).count()
distintos = spark.table(FACT_SALES).select("transaction_id").distinct().count()
dups      = total - distintos
print(f"transaction_id — total: {total:,} | distintos: {distintos:,} | duplicados: {dups}")
assert dups == 0, "transaction_id no es único en fact_sales"
print("✓ transaction_id único OK")

# COMMAND ----------
# Validación 3 — foreign keys nulas

from pyspark.sql import functions as F

fk_nulls = spark.table(FACT_SALES).select(
    F.sum(F.col("customer_key").isNull().cast("int")).alias("customer_key_nulls"),
    F.sum(F.col("item_key").isNull().cast("int")).alias("item_key_nulls"),
    F.sum(F.col("date_key").isNull().cast("int")).alias("date_key_nulls"),
)
display(fk_nulls)

row = fk_nulls.collect()[0]
assert row["customer_key_nulls"] == 0, f"customer_key tiene {row['customer_key_nulls']} nulos"
assert row["item_key_nulls"]     == 0, f"item_key tiene {row['item_key_nulls']} nulos"
assert row["date_key_nulls"]     == 0, f"date_key tiene {row['date_key_nulls']} nulos"
print("✓ Foreign keys sin nulos OK")

# COMMAND ----------
# Muestra final

display(spark.table(FACT_SALES).limit(10))

Silver         : 11,362 filas
dim_customer   : 25 filas
dim_item       : 200 filas
dim_date       : 1,114 filas
✓ Tabla escrita: workspace.default.fact_sales
Filas Silver    : 11,362
Filas fact_sales: 11,362
✓ Conteo OK
transaction_id — total: 11,362 | distintos: 11,362 | duplicados: 0
✓ transaction_id único OK


customer_key_nulls,item_key_nulls,date_key_nulls
0,0,0


✓ Foreign keys sin nulos OK


transaction_id,customer_key,item_key,date_key,quantity,price_per_unit,total_spent,payment_method,location,discount_applied
TXN_6867343,9,8,20240408,10,18.50,185.00,Digital Wallet,Online,true
TXN_3731986,22,63,20230723,9,29.00,261.00,Digital Wallet,Online,true
TXN_9303719,2,18,20221005,2,21.50,43.00,Credit Card,Online,false
TXN_9458126,6,49,20220507,9,27.50,247.50,Credit Card,Online,false
TXN_4575373,5,173,20221002,7,12.50,87.50,Digital Wallet,Online,false
TXN_3652209,7,85,20230610,8,5.00,40.00,Credit Card,In-store,true
TXN_9728486,23,54,20230426,1,27.50,27.50,Credit Card,In-store,false
TXN_2722661,25,106,20240314,3,36.50,109.50,Cash,Online,false
TXN_8776416,22,146,20241214,9,8.00,72.00,Cash,In-store,true
TXN_5874772,23,141,20230909,7,6.50,45.50,Cash,Online,true
